# MapBiomas Fogo — Export & Vectorization

| 🇧🇷 PT | 🇬🇧 EN | 🇪🇸 ES |
|---|---|---|
| Exporta, monta mosaicos (COG), publica e (opcionalmente) vetoriza as coleções do **MapBiomas Fogo** (multipaís). | Exports, mosaics (COG), publishes and (optionally) vectorizes **MapBiomas Fire** collections (multi-country). | Exporta, monta mosaicos (COG), publica y (opcionalmente) vectoriza las colecciones de **MapBiomas Fuego** (multipaís). |

| | PT | EN | ES |
|---|---|---|---|
| **Export** | GEE → tiles temporários no GCS | GEE → temp tiles in GCS | GEE → tiles temporales en GCS |
| **Mosaic** | Tiles → COG | Tiles → COG | Tiles → COG |
| **Public Mosaic** | Espelha COG no bucket público | Mirrors COG to public bucket | Replica COG en el bucket público |
| **Clean Temp** | Apaga `temp/` | Deletes `temp/` | Elimina `temp/` |
| **Vector GCS** *(opcional/optional)* | Raster → shapefile ZIP | Raster → shapefile ZIP | Raster → shapefile ZIP |
| **Vector GEE** *(opcional/optional)* | ZIP → FeatureCollection | ZIP → FeatureCollection | ZIP → FeatureCollection |
| **Public Vector** *(opcional/optional)* | Espelha ZIP no bucket público | Mirrors ZIP to public bucket | Replica ZIP en el bucket público |

**Uso / Usage / Uso:** execute as células de cima para baixo · run cells top to bottom · ejecute las celdas de arriba a abajo. A interface tem guias de ajuda em PT, EN e ES · the interface has help tabs in PT, EN and ES · la interfaz tiene guías en PT, EN y ES.

# Code


In [ ]:
#@title Install Dependencies
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq gdal-bin python3-gdal > /dev/null 2>&1
!pip install --quiet gcsfs earthengine-api geopandas rasterio psutil ipywidgets

In [ ]:
#@title Clone Repository
# Catalog (config.py): https://github.com/mapbiomas/brazil-fire/blob/main/utils/mapbiomas_fire_export_and_vectorization/config.py
!rm -rf /content/brazil-fire
!git clone -q --depth 1 --branch main https://github.com/mapbiomas/brazil-fire.git /content/brazil-fire

import sys
sys.path.append('/content/brazil-fire/utils')

In [ ]:
#@title Configuration (Countries, Buckets, Themes)
# Country tabs: [] = discover ALL from config.OBJ; or list explicit OBJ codes.
# Available country codes: brasil, argentina, bolivia, chile, colombia, ecuador,
#   indonesia, paraguay, peru, venezuela  (e.g. ["brasil", "ecuador"]).
COUNTRIES     = []                     # [] = all from config.OBJ
SCALE         = 30                     # export resolution (m)
GEE_PROJECT   = "mapbiomas-fire-485203"  # GEE project for upload (credentials/billing)
BUCKET_GCS    = "mapbiomas-fire"       # processing bucket
PUBLIC_BUCKET = "mapbiomas-public"     # public bucket (mirror)
BUCKET_PATH   = "initiatives"          # intermediate folder root (GCS)
EXPORT_FLAG   = "Fire-"

# Theme: fire only.
THEMES = ["fire"]

from google.colab import auth
auth.authenticate_user()

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

# All config is handled at call time -> re-run this cell ALWAYS to propagate.
from mapbiomas_fire_export_and_vectorization import config

# Resolve as abas: [] = descobre do OBJ; lista = so os codigos dados (filtra por THEMES).
COUNTRIES = config.resolve_countries(COUNTRIES, themes=THEMES)
config.GEE_PROJECT   = GEE_PROJECT
config.BUCKET        = BUCKET_GCS
config.PUBLIC_BUCKET = PUBLIC_BUCKET
config.BUCKET_PATH   = BUCKET_PATH
for c in COUNTRIES:
    config.COUNTRIES.setdefault(c, {})["scale"] = SCALE
config.COUNTRIES_AVAILABLE = COUNTRIES

# Apply theme filter to UI tabs (empty list = all themes)
config.THEMES = THEMES

# Vector output alternative (None = default sibling _vectors folder).
# Placeholders: {country}, {COUNTRY}, {theme}, {collection}, {COLLECTION},
# {product}, {raster}, {vectors_folder}
# ex.: "projects/mapbiomas-{COUNTRY}/assets/FIRE/{COLLECTION}/FINAL_PRODUCTS"
# ex.: "projects/mapbiomas-{COUNTRY}/assets/FIRE/CATALOG_01/COLLECTION_06/FINAL_PRODUCTS"
VECTOR_OUTPUT_GEE = None
VECTOR_OUTPUT_GCS = None
config.VECTOR_OUTPUT_GEE = VECTOR_OUTPUT_GEE
config.VECTOR_OUTPUT_GCS = VECTOR_OUTPUT_GCS

# If a country's collection/product changes, override here:
# config.COUNTRIES["brazil"]["image_collection"] = "projects/..."
# config.COUNTRIES["brazil"]["product"]         = "..."

import mapbiomas_fire_export_and_vectorization.export as _export
_export.EXPORT_FLAG = EXPORT_FLAG

## Main Interface



In [ ]:
# Open the interface (country/theme/collection/product navigation + units).
# Click LOAD DATA / SYNC (single button) to load units, discover new bands/units
# and check the status of each step.
# Switch country via TAB. Each tab has its own grid/status.
from mapbiomas_fire_export_and_vectorization.ui import run_ui

ui = run_ui(COUNTRIES)



In [ ]:
#@title Step 1: Export
# Launches Earth Engine tasks; may take time.
# Track progress in GEE: https://code.earthengine.google.com/tasks
# After completion, go back to UI cell and click Sync.
# FORCE_EXPORT = True deletes unit tiles (temp/) before re-exporting.
FORCE_EXPORT = False
from mapbiomas_fire_export_and_vectorization.export import export_selected

export_selected(ui, logger=ui._log, force=FORCE_EXPORT)



In [ ]:
#@title Step 2: Mosaic
# Parallel execution (max 4 workers).
# Only processes units that already have tiles in GCS.
# FORCE_MOSAIC = True deletes existing COG before rebuilding.
FORCE_MOSAIC = False
from mapbiomas_fire_export_and_vectorization.mosaic import mosaic_selected

mosaic_selected(ui, logger=ui._log, force=FORCE_MOSAIC)



In [ ]:
#@title Step 3: Public Mosaic
# Mirrors product COG folders to mapbiomas-public (all units per affected product).
# Idempotent. With grid selection, iterates over ALL affected contexts/products.
# FORCE_PUBLISH_MOSAIC = True overwrites COGs in public bucket.
FORCE_PUBLISH_MOSAIC = False
from mapbiomas_fire_export_and_vectorization.publish import publish_mosaic_all

publish_mosaic_all(ui=ui, logger=ui._log, force=FORCE_PUBLISH_MOSAIC)



In [ ]:
#@title Step 4: Clean Temp
# Deletes the temp/ tiles of the SELECTED units (across all panels/countries).
# No preconditions: independent of Mosaic/Public Mosaic/Vector steps.
# Only touches {BUCKET}/{product}/temp/*.tif patterns — never COGs, ZIPs,
# the public bucket or GEE assets.
from mapbiomas_fire_export_and_vectorization.publish import cleanup_temp_selected

cleanup_temp_selected(ui=ui, logger=ui._log)



## Vectorization (optional — vectorizable products, e.g. `annual_burned`)


In [ ]:
#@title Step 5 (Optional): Vector GCS
# Uses gdal_polygonize (ignores pixel 0).
# Parallel execution (max 4 workers).
# Only processes units that already have mosaic in GCS. Vectorizable products only.
# FORCE_VECTOR = True deletes existing ZIP before re-vectorizing.
FORCE_VECTOR = False
from mapbiomas_fire_export_and_vectorization.vectorize import vectorize_selected

vectorize_selected(ui, logger=ui._log, force=FORCE_VECTOR)



In [ ]:
#@title Step 6 (Optional): Vector GEE
# Sequential execution (earthengine upload table).
# Creates sibling folder (if missing) next to the source raster (mapbiomas-public):
#   projects/mapbiomas-public/assets/{country}/fire/{collection}/{raster}_vectors
# and makes folder + assets public (all_users_can_read).
# Only processes units that already have ZIP in GCS. Vectorizable products only.
# FORCE_UPLOAD = True deletes existing GEE asset before re-upload.
FORCE_UPLOAD = False
from mapbiomas_fire_export_and_vectorization.vectorize import gee_upload_selected

gee_upload_selected(ui, logger=ui._log, force=FORCE_UPLOAD)



In [ ]:
#@title Optional: Make Vectors Public
# For vectors. Run AFTER GEE upload tasks complete.
# Folders are already public; this cell covers individual assets (idempotent).
# With grid selection, applies to ALL affected contexts/products.
from mapbiomas_fire_export_and_vectorization.vectorize import make_vectors_public

make_vectors_public(ui=ui, logger=ui._log)



In [ ]:
#@title Step 7 (Optional): Public Vector
# Mirrors vector ZIP folders to mapbiomas-public. Idempotent.
# With grid selection, iterates over ALL affected contexts/products.
# FORCE_PUBLISH_VECTOR = True overwrites ZIPs in public bucket.
FORCE_PUBLISH_VECTOR = False
from mapbiomas_fire_export_and_vectorization.publish import publish_vector_all

publish_vector_all(ui=ui, logger=ui._log, force=FORCE_PUBLISH_VECTOR)

